# Argus prototype data tutorial 1: accessing images. 

These notebooks use a sample of data from V2 of the Argus Array Deep Field Demonstrator (ADDem; https://www.spiedigitallibrary.org/conference-proceedings-of-spie/14147/1414750/ADDem-the-Argus-deep-field-demonstrator/10.1117/12.3105473.full). ADDemV2 is a robotic system of two production Argus cameras and telescopes, which image a simultaneous 8 sq. deg FoB through a blue band, **β** (365–520 nm), and
a red band, **ρ** (565–825 nm), with 60 s exposures.

1. **The archive and its images** (this notebook): the footprint, the frames
   covering a target, and images from a whole sky segment down to a time
   sequence of small cutouts.
2. **Minipixes and their records**: the small cutouts the pipeline keeps for
   every frame around bright stars, with their data-quality and
   photometric-calibration records.
3. **Light curves**: aperture photometry with photutils, calibrated and
   differential light curves of an eclipsing binary, and a period search.
4. **Flares**: four M-dwarf flares, measured on segment cutouts.

The staged files are FITS, read with [`argex`](https://github.com/argus-hdps/argex),
a Python client to the Argus data archive: `argex.read()` returns the same
objects the archive's live service would. Everything else is numpy, astropy
and photutils. 

## Parameters


In [ ]:
# The target: any star in the staged segment works (see notebook 2 for the list)
TARGET = dict(
    name="Gaia DR3 240827852899491328",
    ra=47.786654, dec=43.945067,  # deg, ICRS
    G=12.727, bp_rp=0.919,        # Gaia DR3 photometry
    gaia_frequency=1.2673242,     # 1/d, gaiadr3.vari_eclipsing_binary
)
SEG = 391749                      # the staged segment holding it, in standard Argus TANSeg format

## Setup

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.visualization import AsinhStretch, ImageNormalize, ZScaleInterval

import argex
from argex import sky, times

warnings.filterwarnings("ignore", category=RuntimeWarning)  # all-NaN slices

GREEK = {"beta": "β", "rho": "ρ"}  # band names in plot labels
plt.rcParams.update({"figure.dpi": 110, "image.origin": "lower"})

# The staged sample. Set ARGUS_SAMPLE to read another copy, such as a download.
SAMPLE = Path(os.environ.get("ARGUS_SAMPLE", "~/s3/schmidt-observatory-system/argus/prototype_v1")).expanduser()
assert (SAMPLE / "manifest.json").is_file(), f"no sample at {SAMPLE}: set ARGUS_SAMPLE"

ra, dec = TARGET["ra"], TARGET["dec"]
FIELD = SAMPLE / f"seg_{SEG}"

A few conventions for the dataset:

* **Times** are *Argus epochs*: TT seconds since 2025-01-01T00:00:00 TT, at
  each exposure's midpoint. `argex.times` converts them to and from ISO
  strings, MJD and astropy `Time`.
* **Segments** are the archive's resampled sky tiles: 0.5° north-up TAN images,
  2048 × 2048 pixels at 0.879″/pixel. Every Argus full-frame image is reprojected onto the
  segments it covers, so a segment holds a time series of aligned images.
* **Frames** are identified by a *frame key*, packing the camera's telescope
  configuration code and the epoch. 
* **Pixel values** are electrons per second in each segment pixel. The
  reprojection conserves flux to <1%, so summing pixels over an aperture gives a
  source's count rate.

## 1. Sample archive data

`segments.fits` includes every segment and metadata its id, center, frame count
and epoch range. Plotting the centers on an all-sky Mollweide projection
shows the footprint, with RA increasing to the left as on the sky. The star
marks the segment staged in full for these notebooks.

In [ ]:
import json

import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.table import Table

segs = Table.read(SAMPLE / "segments.fits")
print(f"{len(segs)} segments, {segs['n_frames'].sum():,} segment-frames")
print("epochs:", times.to_iso(segs["epoch_min"].min())[:16], "to",
      times.to_iso(segs["epoch_max"].max())[:16], "UTC")


def sky_xy(ra, dec):
    # degrees -> Mollweide axes coordinates (radians), RA 0 at the center
    # and increasing to the left
    return -np.radians((np.asarray(ra) + 180) % 360 - 180), np.radians(dec)


fig = plt.figure(figsize=(10, 5.4))
ax = fig.add_subplot(projection="mollweide")
plane = SkyCoord(l=np.arange(0, 360, 0.5) * u.deg, b=0 * u.deg, frame="galactic").icrs
gx, gy = sky_xy(plane.ra.deg, plane.dec.deg)
order = np.argsort(gx)
ax.plot(gx[order], gy[order], color="0.6", lw=1, label="Galactic plane")
sc = ax.scatter(*sky_xy(segs["ra"], segs["dec"]), c=segs["n_frames"], s=3, cmap="viridis", zorder=3)
staged = segs[segs["seg"] == 391749]
ax.plot(*sky_xy(staged["ra"], staged["dec"]), "*", color="tab:orange", ms=14, mec="k", zorder=4,
        label="staged segment")
ticks = np.arange(-150, 151, 30)
ax.set_xticks(np.radians(ticks), [f"{-t % 360}°" for t in ticks], fontsize=8)
ax.grid(alpha=0.3)
ax.set_title("Segments with data", pad=12)
ax.legend(loc="lower right", fontsize=8)
fig.colorbar(sc, orientation="horizontal", shrink=0.5, pad=0.06, label="frames per segment")
plt.show()

## 2. The frames covering a target

Our target, set in the parameters at the top, is the eclipsing binary
Gaia DR3 240827852899491328. 

The staged segment comes with a list of every frame's epoch, frame key
and telescope configuration code. 

In [ ]:
frames = Table.read(FIELD / "frames.fits")
bands = {b["name"]: b for b in json.load(open(SAMPLE / "bands.json"))}
band_of = {c: name for name, b in bands.items() for c in b["configs"]}
frames["band"] = [band_of.get(int(c), "?") for c in frames["telescope_config"]]
print(len(frames), "frames:", {name: int((frames["band"] == name).sum()) for name in bands})

utc = times.to_datetime(np.asarray(frames["epoch"]))
fig, ax = plt.subplots(figsize=(9, 2.2))
for band, color in (("beta", "tab:blue"), ("rho", "tab:red")):
    sel = np.asarray(frames["band"] == band)
    ax.plot(np.asarray(utc)[sel], np.full(sel.sum(), GREEK[band]), "|", color=color, ms=12)
ax.set(title=f"Frames of segment {SEG}", xlabel="UTC", ylim=(-0.6, 1.6))

fig.autofmt_xdate()
plt.show()

## 3. Loading the images

### A whole segment

The sample holds one frame of the segment in each band, at full resolution:
2048² pixels. `argex.read()` returns a `Cutout`: the pixels, the
frame identity and the WCS, so matplotlib can draw sky coordinates.
Pixels the frame did not cover are NaN, which is the blank band on the
right: this camera's field ends there.

In [ ]:
images = {p.name.split("_")[0]: argex.read(p) for p in sorted((FIELD / "images").glob("*.fits"))}
img = images["rho"]
print(img.frame.camera, img.band, img.time.utc.isot, "frame key", img.frame.frame_key)

norm = ImageNormalize(img.data, interval=ZScaleInterval())
fig = plt.figure(figsize=(6.5, 6.5))
ax = fig.add_subplot(projection=img.wcs)
ax.imshow(img.data, norm=norm, cmap="gray")
tx, ty = img.radec_to_pixel(ra, dec)
ax.plot(tx, ty, "o", mfc="none", mec="tab:orange", ms=14, mew=1.5)
ax.set(xlabel="RA", ylabel="Dec", title=f"Segment {SEG}, ρ, {img.time.utc.iso[:16]} UTC")
plt.show()

### A cutout in each band

The two cameras expose together, so the same epoch gives a β and a ρ view.
Slicing the arrays around the target gives a 96-pixel cutout of each. This is commissioning data, pre-final optical alignment. 

In [ ]:
half = 48
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, band in zip(axes, ("beta", "rho")):
    im = images[band]
    cx, cy = (int(round(v)) for v in im.radec_to_pixel(ra, dec))
    cut = im.data[cy - half : cy + half, cx - half : cx + half]
    ax.imshow(cut, norm=ImageNormalize(cut, interval=ZScaleInterval()), cmap="gray")
    ax.plot(half, half, "o", mfc="none", mec="tab:orange", ms=16)
    ax.set_title(f"{GREEK[band]}  {im.time.utc.iso[11:19]} UTC")
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

### A time sequence

A `CutoutStack` is the same window from many frames: a cube of per-image planes.  Here, half an hour of
β frames around the target:

In [ ]:
stack = argex.read(FIELD / "stacks" / "target_beta.fits")
print(stack.data.shape, "planes")
norm = ImageNormalize(np.nanmedian(stack.data, axis=0), stretch=AsinhStretch(0.02))
n = min(len(stack), 10)
fig, axes = plt.subplots(1, n, figsize=(1.6 * n, 1.9))
for ax, cut in zip(axes, list(stack)[:n]):
    ax.imshow(cut.data, norm=norm, cmap="gray")
    ax.set_title(cut.time.utc.iso[11:16], fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("β, UTC", fontsize=9, y=1.02)
plt.show()

The header carries the frame's provenance and time:

In [ ]:
from astropy.io import fits

header = fits.getheader(FIELD / "stacks" / "target_beta.fits")
for key in ("TANSEGID", "NFRAMES", "BAND", "CAMID", "TIMESYS", "ARGEPMIN", "ARGEPMAX", "BUNIT", "FLUXCONS"):
    print(f"{key:<9}= {header[key]!r:<22} / {header.comments[key]}")